In [1]:
%load_ext autoreload
%autoreload 2

import copy
from pathlib import Path
from typing import Any

import PIL.Image
import safetensors.torch
import torch
from torch import nn
import transformers

import weight_formats.quantisation as Q
import weight_formats.fit as F
import weight_formats.quantisation_training as T

import squashedtensors as sqt
import train

OUT = Path("out/models/dev")
OUT.mkdir(exist_ok=True, parents=True)

In [ ]:
%%sh
aws s3 cp s3://graphcore-research-us-east-1/2024-10-squashedllama/models/test.jpg out/
# aws s3 cp s3://graphcore-research-us-east-1/2024-10-squashedllama/checkpoints/volcanic-dawn-1773.safetensors out/
aws s3 cp s3://graphcore-research-us-east-1/2024-10-squashedllama/checkpoints/proud-sponge-1878.safetensors out/

download: s3://graphcore-research-us-east-1/2024-10-squashedllama/checkpoints/proud-sponge-1878.safetensors to out/proud-sponge-1878.safetensors


## Text: Llama-3.2-1B-Instruct

In [2]:
model = transformers.LlamaForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct", torch_dtype=torch.bfloat16, device_map="cuda:0")
processor = transformers.AutoProcessor.from_pretrained(model.config._name_or_path)

def show(qmodel: Any) -> None:
    prompt = "Q: Explain Vector Quantisation & Lloyd's Algorithm? A:"
    n_tokens = 100
    args = dict(max_new_tokens=n_tokens, do_sample=False, temperature=None, top_p=None, pad_token_id=processor.eos_token_id)
    inputs = processor(prompt, return_tensors="pt")
    print(f"=== {prompt}")
    print(f"\n### ORIGINAL ###")
    print(processor.decode(model.generate(**inputs.to(model.device), **args)[0]))
    print(f"\n### QUANTISED ###")
    print(processor.decode(qmodel.generate(**inputs.to(qmodel.device), **args)[0]))
    print()

In [ ]:
sqt.save(model, processor, None, OUT / "text-bf16.sqt")

### CHANNEL_INT8

In [112]:
qmodel = copy.deepcopy(model)
T.convert(qmodel, train.FMT_CHANNEL_INT8, "parameter",
          clip_gradient=False, error_weight=None, activation_fmt=train.FMT_CHANNEL_INT8)
show(qmodel)

=== Q: Explain Vector Quantisation & Lloyd's Algorithm? A:

### ORIGINAL ###
<|begin_of_text|>Q: Explain Vector Quantisation & Lloyd's Algorithm? A: Vector Quantization is a method of representing a continuous signal as a discrete set of values. It is used to reduce the amount of data required to represent a signal. Lloyd's Algorithm is a method of finding the best approximation of a set of data points in a high-dimensional space. It is used in various applications such as image compression, data compression, and machine learning. Vector Quantization: Vector Quantization is a method of representing a continuous signal as a discrete set of values. It is used to

### QUANTISED ###
<|begin_of_text|>Q: Explain Vector Quantisation & Lloyd's Algorithm? A: Vector Quantization is a method of representing a continuous signal as a discrete set of values. It is a technique used in image and audio compression. Lloyd's Algorithm is a method of finding the k most similar vectors in a set of k-dimens

In [26]:
sqt.save(qmodel, processor, None, OUT / "text-int8.sqt")

### CHANNEL_S3D8

In [4]:
qmodel = copy.deepcopy(model)
fmt_spec = {k: train.FMT_CHANNEL_S3D8 if "self_attn." in k else train.FMT_CHANNEL_INT8
            for k, v in model.named_parameters()
            if v.ndim == 2}
T.convert(qmodel, fmt_spec, "parameter",
          clip_gradient=False, error_weight=None, activation_fmt=train.FMT_CHANNEL_INT8, progress=True)
show(qmodel)

Quantising: 2it [00:00, 1608.55it/s, model.layers.0.self_attn.q_proj]

Quantising: 114it [00:06, 17.74it/s, lm_head]                         


=== Q: Explain Vector Quantisation & Lloyd's Algorithm? A:

### ORIGINAL ###
<|begin_of_text|>Q: Explain Vector Quantisation & Lloyd's Algorithm? A: Vector Quantization is a method of representing a continuous signal as a discrete set of values. It is used to reduce the amount of data required to represent a signal. Lloyd's Algorithm is a method of finding the best approximation of a set of data points in a high-dimensional space. It is used in various applications such as image compression, data compression, and machine learning. Vector Quantization: Vector Quantization is a method of representing a continuous signal as a discrete set of values. It is used to

### QUANTISED ###
<|begin_of_text|>Q: Explain Vector Quantisation & Lloyd's Algorithm? A: A) Vector Quantisation & Lloyd's Algorithm B) Vector Quantisation & Lloyd's Algorithm C) Vector Quantisation & Lloyd's Algorithm D) Vector Quantisation & Lloyd's Algorithm No, the correct answer is: A) Vector Quantisation & Lloyd's Algorith

In [6]:
sqt.save(qmodel, processor, None, OUT / "text-s3d8-attn-only.sqt")

## Vision: Llama-3.2-11B-Vision-Instruct

In [7]:
model = transformers.MllamaForConditionalGeneration.from_pretrained(
    "meta-llama/Llama-3.2-11B-Vision-Instruct", torch_dtype=torch.bfloat16, device_map="cuda:0")
processor = transformers.AutoProcessor.from_pretrained(model.config._name_or_path)

def show(qmodel: Any) -> None:
    img = PIL.Image.open("out/test.jpg")
    scale = model.config.vision_config.image_size / max(img.width, img.height)
    img = img.resize((int(img.width * scale), int(img.height * scale)))
    prompt = "<|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\\n<|start_header_id|>assistant<|end_header_id|>"
    n_tokens = 30

    inputs = processor(img, prompt, return_tensors="pt")
    print(f"=== {prompt}")
    print(f"\n### ORIGINAL ###")
    print(processor.decode(model.generate(**inputs.to(model.device), max_new_tokens=n_tokens, do_sample=False, temperature=None, top_p=None)[0]))
    print(f"\n### QUANTISED ###")
    print(processor.decode(qmodel.generate(**inputs.to(qmodel.device), max_new_tokens=n_tokens, do_sample=False, temperature=None, top_p=None)[0]))
    print()

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [56]:
sqt.save(model, processor.tokenizer, processor.image_processor, OUT / "vision-bf16.sqt")

/root/squashed-llama/training/squashedtensors.py:181: UserWarning: The tokenizer does not set `ignore_merges` (which matches full tokens first), but our codebase always behaves as if `ignore_merges` is `True`
  warnings.warn(


### CHANNEL_INT8

In [ ]:
qmodel = copy.deepcopy(model)
T.convert(qmodel, train.FMT_CHANNEL_INT8, "parameter",
          clip_gradient=False, error_weight=None, activation_fmt=train.FMT_CHANNEL_INT8)
show(qmodel)

=== <|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>

### ORIGINAL ###
<|begin_of_text|><|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>

The person with the laptop is wearing a red shirt. The person with the laptop is wearing a red shirt. The person with the laptop is wearing

### QUANTISED ###
<|begin_of_text|><|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>

The person with the laptop is wearing a red shirt.<|eot_id|>



In [77]:
sqt.save(qmodel, processor.tokenizer, processor.image_processor, OUT / "vision-int8.sqt")

/root/squashed-llama/training/squashedtensors.py:181: UserWarning: The tokenizer does not set `ignore_merges` (which matches full tokens first), but our codebase always behaves as if `ignore_merges` is `True`
  warnings.warn(


### CHANNEL_S3D8

In [8]:
qmodel = copy.deepcopy(model)
fmt_spec = {k: train.FMT_CHANNEL_S3D8 if any(sub in k for sub in ["self_attn.", "cross_attn."]) else train.FMT_CHANNEL_INT8  # "mlp.",
            for k, v in model.named_parameters()
            if v.ndim == 2}
T.convert(qmodel, fmt_spec, "parameter",
          clip_gradient=False, error_weight=None, activation_fmt=train.FMT_CHANNEL_INT8, progress=True)

Quantising: 2it [00:00, 1316.69it/s, vision_model.pre_tile_positional_embedding.embedding]   

Quantising: 526it [00:42, 12.50it/s, multi_modal_projector]                                    


In [9]:
show(qmodel)

=== <|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>

### ORIGINAL ###
<|begin_of_text|><|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>

The person with the laptop is wearing a red shirt. The person with the laptop is wearing a red shirt. The person with the laptop is wearing

### QUANTISED ###
<|begin_of_text|><|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>

The person with the laptop is wearing a red shirt.<|eot_id|>



In [10]:
sqt.save(qmodel, processor.tokenizer, processor.image_processor, OUT / "vision-s3d8-attn-only.sqt")

/root/squashed-llama/training/squashedtensors.py:181: UserWarning: The tokenizer does not set `ignore_merges` (which matches full tokens first), but our codebase always behaves as if `ignore_merges` is `True`
  warnings.warn(


### volcanic-dawn-1773 (CHANNEL_S3D8 + QAT)

In [ ]:
qmodel = copy.deepcopy(model)
T.load_convert(qmodel, safetensors.torch.load_file("out/volcanic-dawn-1773.safetensors"))

In [ ]:
show(qmodel)

=== <|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>

### ORIGINAL ###
<|begin_of_text|><|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>

The person with the laptop is wearing a red shirt. The person with the laptop is wearing a red shirt. The person with the laptop is wearing

### QUANTISED ###
<|begin_of_text|><|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|start_header_id|><|st

In [ ]:
sqt.save(qmodel, processor.tokenizer, processor.image_processor, OUT / "vision-s3d8-volcanic-dawn-1773.sqt")

### proud-sponge-1878 (CHANNEL_S3D8 + QAT)

In [8]:
qmodel = copy.deepcopy(model)
T.load_convert(qmodel, safetensors.torch.load_file("out/proud-sponge-1878.safetensors"))

In [9]:
show(qmodel)

=== <|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>

### ORIGINAL ###
<|begin_of_text|><|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>

The person with the laptop is wearing a red shirt. The person with the laptop is wearing a red shirt. The person with the laptop is wearing

### QUANTISED ###
<|begin_of_text|><|image|><|begin_of_text|>What colour shirt is the person with the laptop wearing?<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>

The person with the laptop is wearing a purple shirt.<|eot_id|>



In [10]:
sqt.save(qmodel, processor.tokenizer, processor.image_processor, OUT / "vision-s3d8-proud-sponge-1878.sqt")

/root/squashed-llama/training/squashedtensors.py:181: UserWarning: The tokenizer does not set `ignore_merges` (which matches full tokens first), but our codebase always behaves as if `ignore_merges` is `True`
  warnings.warn(


In [13]:
!ls -lh out/models/dev/vision-s3d8-proud-sponge-1878.sqt

-rw-r--r--. 1 root root 3.5G Apr 24 09:57 out/models/dev/vision-s3d8-proud-sponge-1878.sqt


## Sync models

In [12]:
!aws s3 sync out/models/dev/ s3://graphcore-research-us-east-1/2024-10-squashedllama/models/dev/

upload: out/models/dev/vision-s3d8-proud-sponge-1878.sqt to s3://graphcore-research-us-east-1/2024-10-squashedllama/models/dev/vision-s3d8-proud-sponge-1878.sqt
